# Music Annotation Project V6: Leakage-Safe Mask Policy Optimisation

**Runtime:** CPU. Select **Runtime > Run all**.

This experimental notebook optimises reconstruction-mask policy without allowing the clean target to influence production simulation for its own held-out page.

Core safeguards:
- candidate masks use only the observed page and corrected seed mask;
- the production kernel has no clean-page argument;
- candidate outputs are frozen and hashed before clean-page evaluation;
- leave-one-page-out selection chooses the policy for each held-out page using other pages only;
- every reconstruction is unchanged outside its candidate mask;
- all policies are compared on the same fixed evaluation ROI per page;
- an oracle diagnostic is exported separately and never participates in policy selection.

The first experiment deliberately holds reconstruction fixed at biharmonic. It asks whether mask policy improves results before adding more reconstruction methods.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless scikit-image pandas matplotlib


## Configuration and durable paths


In [ ]:
import cv2, json, hashlib, shutil, math, time, inspect, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone
from skimage.restoration import inpaint as sk_inpaint
from skimage.metrics import structural_similarity as ssim

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
SCHEMA=6
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
CLEAN_DIR=ROOT/'source'/'clean'
OBSERVED_DIR=ROOT/'aligned'
SEED_MASK_DIR=ROOT/'masks'/'corrected'
OUTPUT_ROOT=ROOT/'reconstruction_mask_benchmark_v6'
PERSISTENT_CACHE=OUTPUT_ROOT/'persistent_cache'
RUN_NAME=datetime.now().strftime('%Y%m%d_%H%M%S')
OUT=OUTPUT_ROOT/RUN_NAME
TABLES=OUT/'tables'; CONTACTS=OUT/'contact_sheets'; FROZEN=OUT/'frozen_production'
LOCAL=Path('/content/reconstruction_mask_benchmark_v6')
LOCAL_OBSERVED=LOCAL/'observed'; LOCAL_MASK=LOCAL/'seed_masks'; LOCAL_CLEAN=LOCAL/'clean'
for d in [OUT,TABLES,CONTACTS,FROZEN,PERSISTENT_CACHE,LOCAL_OBSERVED,LOCAL_MASK,LOCAL_CLEAN]: d.mkdir(parents=True,exist_ok=True)

CROP_MARGIN=48
EVALUATION_MARGIN=8
MIN_COMPONENT_AREA=8
MIN_EVALUATION_COMPONENT_AREA=20
BOOTSTRAP_SAMPLES=2000
FORCE_REBUILD=False
MAX_CONTACT_PAGES=20

# Start deliberately small. Expand only after inspecting the first run.
DILATIONS=[0,1,2,3]
CLOSING_RADII=[0,1]
EDGE_PROTECTION_STRENGTHS=[0.0,0.5,1.0]
ADAPTIVE_EXTRA_DILATION=[0,1]

# Composite loss. Lower is better. These are experimental weights, not established facts.
LOSS_WEIGHTS={
    'mae':1.0,
    'one_minus_ssim':35.0,
    'one_minus_ink_f1':20.0,
    'one_minus_edge_f1':15.0,
    'mask_growth_fraction':8.0,
    'catastrophic_darkening':30.0,
}


## Discovery, staging and fingerprints


In [ ]:
def image_files(folder):
    folder=Path(folder)
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED) if folder.exists() else []
def maps(folder): return {p.stem:p for p in image_files(folder)}
def md5_file(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()
def array_md5(a): return hashlib.md5(np.ascontiguousarray(a).tobytes()).hexdigest()
def object_hash(x): return hashlib.md5(json.dumps(x,sort_keys=True).encode()).hexdigest()
def copy_if_changed(src,dst):
    dst=Path(dst)
    if not dst.exists() or src.stat().st_size!=dst.stat().st_size or src.stat().st_mtime_ns!=dst.stat().st_mtime_ns:
        shutil.copy2(src,dst)

def find_local(folder,page):
    hits=[p for p in image_files(folder) if p.stem.lower()==page.lower()]
    return hits[0] if len(hits)==1 else None

clean_map=maps(CLEAN_DIR); observed_map=maps(OBSERVED_DIR); seed_map=maps(SEED_MASK_DIR)
pages=sorted(set(clean_map)&set(observed_map)&set(seed_map))
if len(pages)<3: raise RuntimeError('V6 requires at least three matched clean/observed/corrected-mask pages.')
for page in pages:
    for src,dst_dir in [(clean_map[page],LOCAL_CLEAN),(observed_map[page],LOCAL_OBSERVED),(seed_map[page],LOCAL_MASK)]:
        copy_if_changed(src,dst_dir/(page+src.suffix.lower()))
print('Matched controlled pages:',len(pages))
print('Output:',OUT)


## Production kernel: no clean target is accepted


In [ ]:
def binary(mask): return np.where(mask>0,255,0).astype(np.uint8)

def remove_small(mask,min_area=MIN_COMPONENT_AREA):
    n,labels,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    out=np.zeros_like(mask)
    for i in range(1,n):
        if stats[i,cv2.CC_STAT_AREA]>=min_area: out[labels==i]=255
    return out

def visible_structure(observed,seed_mask):
    visible=observed.copy(); visible[seed_mask>0]=255
    ink=visible<190
    edges=cv2.dilate(cv2.Canny(visible,70,170),np.ones((3,3),np.uint8))>0
    long=cv2.morphologyEx(ink.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    staff=cv2.dilate(long.astype(np.uint8),np.ones((3,7),np.uint8))>0
    return ink,edges,staff

def make_production_mask(observed,seed_mask,policy):
    # IMPORTANT: production-visible inputs only.
    seed=remove_small(binary(seed_mask))
    radius=int(policy['dilation'])
    mask=seed.copy()
    if radius>0:
        kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*radius+1,2*radius+1))
        mask=cv2.dilate(mask,kernel)
    close=int(policy['closing'])
    if close>0:
        kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*close+1,2*close+1))
        mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,kernel)
    ink,edge,staff=visible_structure(observed,seed)
    extra=int(policy['adaptive_extra'])
    if extra>0:
        extra_mask=cv2.dilate(mask,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*extra+1,2*extra+1)))
        # Expand more only where the observed neighbourhood is relatively structure-poor.
        protected=cv2.dilate((edge|staff).astype(np.uint8),np.ones((5,5),np.uint8))>0
        mask=np.where((extra_mask>0)&~protected,255,mask).astype(np.uint8)
    strength=float(policy['edge_protection'])
    if strength>0:
        protected=cv2.dilate((edge|staff).astype(np.uint8),np.ones((3,3),np.uint8))>0
        removable=(mask>0)&~(seed>0)&protected
        if strength>=1.0:
            mask[removable]=0
        else:
            # Deterministic partial protection using distance from the seed.
            dist=cv2.distanceTransform((seed==0).astype(np.uint8),cv2.DIST_L2,5)
            mask[removable&(dist>max(1.0,radius*strength))]=0
    mask[seed>0]=255
    return remove_small(mask)

def padded_components(mask,margin=CROP_MARGIN):
    n,labels,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    h,w=mask.shape; boxes=[]
    for i in range(1,n):
        x,y,bw,bh,area=stats[i]
        if area>=MIN_COMPONENT_AREA:
            boxes.append((max(0,x-margin),max(0,y-margin),min(w,x+bw+margin),min(h,y+bh+margin)))
    changed=True
    while changed:
        changed=False; merged=[]
        while boxes:
            ax0,ay0,ax1,ay1=boxes.pop(); rest=[]
            for bx0,by0,bx1,by1 in boxes:
                if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):
                    ax0=min(ax0,bx0); ay0=min(ay0,by0); ax1=max(ax1,bx1); ay1=max(ay1,by1); changed=True
                else: rest.append((bx0,by0,bx1,by1))
            boxes=rest; merged.append((ax0,ay0,ax1,ay1))
        boxes=merged
    return boxes

def reconstruct_production(observed,mask):
    result=observed.copy()
    for x0,y0,x1,y1 in padded_components(mask):
        image=observed[y0:y1,x0:x1]; local_mask=mask[y0:y1,x0:x1]>0
        if not np.any(local_mask): continue
        candidate=sk_inpaint.inpaint_biharmonic(image.astype(np.float32)/255.0,local_mask,channel_axis=None)
        candidate=np.clip(candidate*255,0,255).astype(np.uint8)
        target=result[y0:y1,x0:x1]; target[local_mask]=candidate[local_mask]
    assert np.array_equal(result[mask==0],observed[mask==0])
    return result

# Mechanical interface check: the production functions cannot receive clean/GT inputs.
for fn in [make_production_mask,reconstruct_production]:
    forbidden={'clean','ground_truth','target','reference'} & set(inspect.signature(fn).parameters)
    if forbidden: raise RuntimeError(f'Leakage-prone production signature: {fn.__name__} {forbidden}')
print('Production-kernel signature gate passed.')


## Candidate policy grid


In [ ]:
POLICIES=[]
for dilation,closing,protection,extra in itertools.product(DILATIONS,CLOSING_RADII,EDGE_PROTECTION_STRENGTHS,ADAPTIVE_EXTRA_DILATION):
    policy={'dilation':dilation,'closing':closing,'edge_protection':protection,'adaptive_extra':extra}
    policy['policy_id']='p_'+object_hash(policy)[:10]
    POLICIES.append(policy)
policy_df=pd.DataFrame(POLICIES).drop_duplicates('policy_id').sort_values('policy_id').reset_index(drop=True)
policy_df.to_csv(TABLES/'policy_grid.csv',index=False)
print('Candidate policies:',len(policy_df))
display(policy_df)


## Phase A: freeze production outputs before evaluation


In [ ]:
production_rows=[]
for page_no,page in enumerate(pages,1):
    observed_path=find_local(LOCAL_OBSERVED,page); seed_path=find_local(LOCAL_MASK,page)
    observed=cv2.imread(str(observed_path),0); seed=cv2.imread(str(seed_path),0)
    if observed is None or seed is None or observed.shape!=seed.shape: raise RuntimeError(f'{page}: invalid observed/seed pair')
    source_identity={'observed':md5_file(observed_path),'seed':md5_file(seed_path),'schema':SCHEMA,'margin':CROP_MARGIN}
    for _,policy_row in policy_df.iterrows():
        policy=policy_row.to_dict(); pid=policy['policy_id']
        fingerprint=object_hash({'source':source_identity,'policy':policy})
        cache=PERSISTENT_CACHE/f'{page}_{pid}_{fingerprint}.npz'
        if cache.exists() and not FORCE_REBUILD:
            with np.load(cache) as z: mask=z['mask']; reconstruction=z['reconstruction']
            status='HIT'
        else:
            mask=make_production_mask(observed,seed,policy)
            reconstruction=reconstruct_production(observed,mask)
            np.savez(cache,mask=mask,reconstruction=reconstruction)
            status='MISS'
        assert np.array_equal(reconstruction[mask==0],observed[mask==0])
        frozen_dir=FROZEN/page; frozen_dir.mkdir(parents=True,exist_ok=True)
        mask_path=frozen_dir/f'{pid}_mask.png'; recon_path=frozen_dir/f'{pid}_reconstruction.png'
        cv2.imwrite(str(mask_path),mask); cv2.imwrite(str(recon_path),reconstruction)
        production_rows.append({'page':page,'policy_id':pid,'cache':status,'fingerprint':fingerprint,
                                'observed_md5':source_identity['observed'],'seed_md5':source_identity['seed'],
                                'mask_md5':array_md5(mask),'reconstruction_md5':array_md5(reconstruction),
                                'mask_pixels':int(np.sum(mask>0)),'seed_pixels':int(np.sum(seed>0)),
                                'mask_path':str(mask_path),'reconstruction_path':str(recon_path)})
    print(f'Frozen production outputs {page_no}/{len(pages)}: {page}')
production_df=pd.DataFrame(production_rows)
production_df.to_csv(TABLES/'frozen_production_manifest.csv',index=False)
print('Phase A complete. Clean pages have not been loaded by the production loop.')


## Phase B: sealed evaluation with a fixed ROI


In [ ]:
def printed_ink(image):
    return cv2.adaptiveThreshold(image,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edge_map(image): return cv2.Canny(image,70,170)>0
def tolerant_f1(reference,prediction,region,tolerance=1):
    r=reference&region; p=prediction&region
    if r.sum()==0 and p.sum()==0: return 1.0
    if r.sum()==0: return 0.0
    k=np.ones((2*tolerance+1,2*tolerance+1),np.uint8)
    rd=cv2.dilate(r.astype(np.uint8),k)>0; pd=cv2.dilate(p.astype(np.uint8),k)>0
    precision=float(np.sum(p&rd)/max(p.sum(),1)); recall=float(np.sum(r&pd)/max(r.sum(),1))
    return float(2*precision*recall/max(precision+recall,1e-12))
def fixed_evaluation_roi(seed):
    kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*EVALUATION_MARGIN+1,2*EVALUATION_MARGIN+1))
    return cv2.dilate(binary(seed),kernel)>0
def classify_gt(clean,roi):
    ink=printed_ink(clean)
    long=cv2.morphologyEx(ink.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    staff=cv2.dilate(long.astype(np.uint8),np.ones((3,7),np.uint8))>0
    staff_near=cv2.dilate(staff.astype(np.uint8),np.ones((5,9),np.uint8))>0
    return {'all':roi,'background':roi&~ink,'staff':roi&staff,'symbol':roi&ink&~staff_near,'mixed':roi&ink&staff_near}

def evaluate_candidate(clean,observed,seed,mask,reconstruction,region):
    if not np.any(region): return None
    delta=clean[region].astype(np.float32)-reconstruction[region].astype(np.float32)
    mae=float(np.mean(np.abs(delta))); mse=float(np.mean(delta*delta))
    _,sm=ssim(clean,reconstruction,data_range=255,full=True)
    clean_ink=printed_ink(clean); recon_ink=printed_ink(reconstruction)
    ink_f1=tolerant_f1(clean_ink,recon_ink,region,1)
    edge_f1=tolerant_f1(edge_map(clean),edge_map(reconstruction),region,1)
    growth=max(0,int(np.sum(mask>0))-int(np.sum(seed>0)))/max(int(np.sum(seed>0)),1)
    dark_added=np.sum((reconstruction<140)&(clean>=210)&region)/max(np.sum(region),1)
    loss=(LOSS_WEIGHTS['mae']*mae+
          LOSS_WEIGHTS['one_minus_ssim']*(1-float(sm[region].mean()))+
          LOSS_WEIGHTS['one_minus_ink_f1']*(1-ink_f1)+
          LOSS_WEIGHTS['one_minus_edge_f1']*(1-edge_f1)+
          LOSS_WEIGHTS['mask_growth_fraction']*growth+
          LOSS_WEIGHTS['catastrophic_darkening']*float(dark_added))
    return {'pixels':int(np.sum(region)),'mae':mae,'psnr':float(20*np.log10(255/math.sqrt(max(mse,1e-9)))),
            'ssim':float(sm[region].mean()),'ink_f1':ink_f1,'edge_f1':edge_f1,
            'mask_growth_fraction':float(growth),'catastrophic_darkening':float(dark_added),'loss':float(loss)}

evaluation_rows=[]
for page_no,page in enumerate(pages,1):
    clean=cv2.imread(str(find_local(LOCAL_CLEAN,page)),0)
    observed=cv2.imread(str(find_local(LOCAL_OBSERVED,page)),0)
    seed=cv2.imread(str(find_local(LOCAL_MASK,page)),0)
    if clean is None or observed is None or seed is None or clean.shape!=observed.shape or seed.shape!=clean.shape:
        raise RuntimeError(f'{page}: invalid evaluation inputs')
    roi=fixed_evaluation_roi(seed); regions=classify_gt(clean,roi)
    for _,record in production_df[production_df.page==page].iterrows():
        mask=cv2.imread(record.mask_path,0); reconstruction=cv2.imread(record.reconstruction_path,0)
        if array_md5(mask)!=record.mask_md5 or array_md5(reconstruction)!=record.reconstruction_md5:
            raise RuntimeError(f'{page} {record.policy_id}: frozen-output hash mismatch')
        for region_name,region in regions.items():
            values=evaluate_candidate(clean,observed,seed,mask,reconstruction,region)
            if values: evaluation_rows.append({'page':page,'policy_id':record.policy_id,'region':region_name,**values})
    print(f'Evaluated {page_no}/{len(pages)}: {page}')
evaluation_df=pd.DataFrame(evaluation_rows)
evaluation_df.to_csv(TABLES/'candidate_metrics.csv',index=False)
print('Evaluation rows:',len(evaluation_df))


## Leave-one-page-out policy selection


In [ ]:
all_scores=evaluation_df[evaluation_df.region=='all'][['page','policy_id','loss']]
loocv=[]
for held_out in pages:
    development=all_scores[all_scores.page!=held_out]
    ranking=development.groupby('policy_id',as_index=False).agg(development_loss=('loss','mean'),development_median=('loss','median'))
    ranking=ranking.sort_values(['development_loss','development_median','policy_id'])
    winner=ranking.iloc[0]
    test=all_scores[(all_scores.page==held_out)&(all_scores.policy_id==winner.policy_id)].iloc[0]
    baseline=all_scores[(all_scores.page==held_out)&(all_scores.policy_id==policy_df.query('dilation==1 and closing==0 and edge_protection==0 and adaptive_extra==0').iloc[0].policy_id)].iloc[0]
    loocv.append({'held_out_page':held_out,'selected_policy_id':winner.policy_id,
                  'development_loss':float(winner.development_loss),'held_out_loss':float(test.loss),
                  'baseline_policy_id':baseline.policy_id,'baseline_held_out_loss':float(baseline.loss),
                  'benefit_vs_current_dilation1':float(baseline.loss-test.loss)})
loocv_df=pd.DataFrame(loocv)
loocv_df.to_csv(TABLES/'loocv_policy_results.csv',index=False)

final_ranking=all_scores.groupby('policy_id',as_index=False).agg(mean_loss=('loss','mean'),median_loss=('loss','median'),pages=('page','nunique')).sort_values(['mean_loss','median_loss','policy_id'])
final_ranking=final_ranking.merge(policy_df,on='policy_id',how='left')
final_ranking.to_csv(TABLES/'final_policy_ranking.csv',index=False)
recommended_policy=final_ranking.iloc[0].to_dict()
(OUT/'recommended_mask_policy.json').write_text(json.dumps(recommended_policy,indent=2))
display(loocv_df)
display(final_ranking.head(20))
print('Recommended development policy:',recommended_policy)


## Oracle diagnostic ceiling: excluded from selection


In [ ]:
# Diagnostic only. The oracle uses clean-vs-observed differences and is prohibited from policy selection.
def oracle_mask(clean,observed,seed):
    clean_bg=cv2.GaussianBlur(clean,(0,0),25); observed_bg=cv2.GaussianBlur(observed,(0,0),25)
    clean_signal=cv2.subtract(clean_bg,clean); observed_signal=cv2.subtract(observed_bg,observed)
    strength=cv2.subtract(observed_signal,clean_signal)
    _,difference=cv2.threshold(strength,12,255,cv2.THRESH_BINARY)
    difference=cv2.morphologyEx(difference,cv2.MORPH_CLOSE,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3)))
    return binary((difference>0)|(seed>0))

oracle_rows=[]
for page in pages:
    clean=cv2.imread(str(find_local(LOCAL_CLEAN,page)),0); observed=cv2.imread(str(find_local(LOCAL_OBSERVED,page)),0); seed=cv2.imread(str(find_local(LOCAL_MASK,page)),0)
    mask=oracle_mask(clean,observed,seed); reconstruction=reconstruct_production(observed,mask); roi=fixed_evaluation_roi(seed)
    values=evaluate_candidate(clean,observed,seed,mask,reconstruction,roi)
    oracle_rows.append({'page':page,'diagnostic_only':True,**values})
oracle_df=pd.DataFrame(oracle_rows)
oracle_df.to_csv(TABLES/'oracle_diagnostic_ceiling.csv',index=False)
print('Oracle diagnostics exported separately and not used in policy selection.')


## Contact sheets for baseline, selected policy and oracle


In [ ]:
selected_map=dict(zip(loocv_df.held_out_page,loocv_df.selected_policy_id))
baseline_id=loocv_df.baseline_policy_id.iloc[0]
for page in pages[:MAX_CONTACT_PAGES]:
    clean=cv2.imread(str(find_local(LOCAL_CLEAN,page)),0); observed=cv2.imread(str(find_local(LOCAL_OBSERVED,page)),0); seed=cv2.imread(str(find_local(LOCAL_MASK,page)),0)
    selected_id=selected_map[page]
    def frozen_image(pid,kind): return cv2.imread(str(FROZEN/page/f'{pid}_{kind}.png'),0)
    base_mask=frozen_image(baseline_id,'mask'); base_recon=frozen_image(baseline_id,'reconstruction')
    sel_mask=frozen_image(selected_id,'mask'); sel_recon=frozen_image(selected_id,'reconstruction')
    omask=oracle_mask(clean,observed,seed); orecon=reconstruct_production(observed,omask)
    panels=[clean,observed,seed,base_mask,base_recon,sel_mask,sel_recon,omask,orecon]
    names=['clean GT','observed','corrected seed','baseline mask','baseline reconstruction','LOOCV mask','LOOCV reconstruction','oracle mask','oracle reconstruction']
    fig,axes=plt.subplots(3,3,figsize=(18,22))
    for ax,image,name in zip(axes.ravel(),panels,names): ax.imshow(image,cmap='gray',vmin=0,vmax=255); ax.set_title(name); ax.axis('off')
    fig.suptitle(f'{page} | selected {selected_id} | baseline {baseline_id}')
    plt.tight_layout(); plt.savefig(CONTACTS/f'{page}_comparison.png',dpi=150,bbox_inches='tight'); plt.close(fig)
print('Contact sheets:',CONTACTS)


## Bootstrap summary and manifest


In [ ]:
def bootstrap_mean_ci(values,seed=42):
    v=np.asarray(values,float); v=v[np.isfinite(v)]
    if not len(v): return np.nan,np.nan,np.nan
    rng=np.random.default_rng(seed); samples=np.array([rng.choice(v,len(v),replace=True).mean() for _ in range(BOOTSTRAP_SAMPLES)])
    return float(v.mean()),float(np.percentile(samples,2.5)),float(np.percentile(samples,97.5))
mean_benefit,ci_low,ci_high=bootstrap_mean_ci(loocv_df.benefit_vs_current_dilation1)
summary={
    'created_utc':datetime.now(timezone.utc).isoformat(),'schema':SCHEMA,'pages':len(pages),'policies':len(policy_df),
    'reconstruction_method':'biharmonic','selection':'leave_one_page_out','fixed_evaluation_margin':EVALUATION_MARGIN,
    'mean_loocv_benefit_vs_current_dilation1':mean_benefit,'benefit_ci95':[ci_low,ci_high],
    'recommended_policy':recommended_policy,
    'guarantees':{'production_functions_accept_no_clean_target':True,'clean_loaded_only_in_phase_b':True,
                  'frozen_outputs_hashed_before_evaluation':True,'outside_candidate_mask_immutable':True,
                  'same_evaluation_roi_across_policies':True,'oracle_excluded_from_selection':True},
    'loss_weights':LOSS_WEIGHTS,
    'outputs':{'candidate_metrics':str(TABLES/'candidate_metrics.csv'),'loocv':str(TABLES/'loocv_policy_results.csv'),
               'ranking':str(TABLES/'final_policy_ranking.csv'),'oracle':str(TABLES/'oracle_diagnostic_ceiling.csv'),
               'recommended_policy':str(OUT/'recommended_mask_policy.json')}
}
(OUT/'benchmark_manifest.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
print('V6 COMPLETE:',OUT)
